In [ ]:
from selenium import webdriver # Navegador
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import undetected_chromedriver as uc

import pandas as pd
import sys
import shutil
import os
import time
import datetime 
import pickle
import re

from openpyxl import load_workbook


In [ ]:
def get_hyperlinks(nome_file):
    wb = load_workbook(nome_file, data_only=False)
    ws = wb.active

    # Find the column index of the desired header
    target_header = "Link"
    col_index = None

    for cell in ws[1]:  # First row = header
        if cell.value == target_header:
            col_index = cell.column  # This gives the numeric column index (1 for A, 2 for B, etc.)
            break

    if col_index is None:
        raise ValueError(f"Header '{target_header}' not found!")

    # Now extract the formulas from that column
    hyperlinks = []
    for row in ws.iter_rows(min_row=2, min_col=col_index, max_col=col_index):
        formula = row[0].value
        if formula and formula.startswith('=HYPERLINK('):
            url = formula.replace('=HYPERLINK("', '').replace('")', '')
            hyperlinks.append(url)
        else:
            hyperlinks.append(None)
    
    return hyperlinks

In [ ]:
lista_zone = [
              "Navigatori",
              "Garbatella",
              "Ostiense",
              "San Paolo",
              "Marconi",
              "Portuense",
              "Villa Bonelli",
              "Magliana",
              "Aventino",
              #"San Saba - Caracalla",
              "Testaccio",
              "Trastevere",
              "Monteverde Vecchio",
              "Monteverde Nuovo",
              "Colle Oppio",
              "Colosseo - Fori Imperiali",
              "Monti"
]

In [ ]:
# Setup Chrome options
options = uc.ChromeOptions()
options.add_argument("--no-sandbox")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--start-maximized")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
options.add_argument("--window-size=1920x1080")

# options.add_argument("--headless")  # Optional: hide the browser

# This automatically finds the correct version of ChromeDriver
driver = uc.Chrome(version_main=135, options=options)

# Add any options you need, e.g.:
# chrome_options.add_argument("--headless")

# Allow automatic downloads to a custom path
driver.command_executor._commands["send_command"] = ("POST", '/session/$sessionId/chromium/send_command')

# Set timeouts
driver.implicitly_wait(2)
driver.set_script_timeout(900)
driver.set_page_load_timeout(900)
driver.maximize_window()


In [ ]:
import math
from selenium.common.exceptions import NoSuchElementException

def close_optional_banner(driver):
    try:
        close_btn = driver.find_element(By.CSS_SELECTOR, "button.ab-close-button")
        close_btn.click()
        time.sleep(0.5)
    except NoSuchElementException:
        pass  # Banner not present, skip
    
def pixels_to_latlon(px_x, px_y, zoom, tile_size=256):
    n = 2 ** zoom
    lon_deg = px_x / (tile_size * n) * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * px_y / (tile_size * n))))
    lat_deg = math.degrees(lat_rad)
    return lat_deg, lon_deg

def center_pixels_to_latlon(px_x, px_y, zoom, tile_size=256):
    scale = tile_size * 2 ** zoom
    lon = px_x / scale * 360.0 - 180.0
    lat_rad = math.pi - 2.0 * math.pi * px_y / scale
    lat = math.degrees(math.atan(math.sinh(lat_rad)))
    return lat, lon


def zoom_to_max(driver, pause=0.5, max_attempts=10):
    for _ in range(max_attempts):
        try:
            zoom_in_btn = driver.find_element(By.CSS_SELECTOR, '.leaflet-control-zoom-in')
            
            # 🔽 Scroll into view before clicking
            driver.execute_script("arguments[0].scrollIntoView({behavior: 'instant', block: 'center'});", zoom_in_btn)
            time.sleep(1)
            if "leaflet-disabled" in zoom_in_btn.get_attribute("class"):
                break  # already at max zoom

            zoom_in_btn.click()
            time.sleep(pause)
            time.sleep(1)
        except NoSuchElementException:
            break  # no zoom-in button found
            
def extract_precise_pin_coordinates(driver):
    zoom_to_max(driver)  # Zoom in first
    
    # Wait and re-find the tile at final zoom
    time.sleep(1)
    # --- Detect zoom/tile from tile URL
    tile_img = driver.find_element(By.CSS_SELECTOR, "img.leaflet-tile")
    tile_url = tile_img.get_attribute("src")
    zoom, tile_x, tile_y = map(int, re.search(r"/(\d+)/(\d+)/(\d+)\.png", tile_url).groups())
    tile_size = 256
    time.sleep(1)
    # --- Determine scale from URL (or fallback to devicePixelRatio)
    scale_match = re.search(r"scale=(\d+)", tile_url)
    scale = float(scale_match.group(1)) if scale_match else driver.execute_script("return window.devicePixelRatio") or 1.0

    # --- Get map offset
    map_pane = driver.find_element(By.CSS_SELECTOR, ".leaflet-map-pane")
    map_offset_match = re.search(r'translate3d\(([-\d.]+)px,\s*([-\d.]+)px', map_pane.get_attribute("style"))
    map_offset_x, map_offset_y = map(lambda x: float(x)/scale, map_offset_match.groups())
    time.sleep(1)
    # --- Marker transform and margins
    marker = driver.find_element(By.CSS_SELECTOR, ".leaflet-marker-icon")
    style = marker.get_attribute("style")
    marker_x, marker_y = map(lambda x: float(x)/scale,
                             re.search(r'translate3d\(([\d.]+)px,\s*([\d.]+)px', style).groups())
    margin_left = float(re.search(r'margin-left:\s*(-?[\d.]+)px', style).group(1))
    margin_top = float(re.search(r'margin-top:\s*(-?[\d.]+)px', style).group(1))
    time.sleep(1)
    # --- Compute pixel location of pin tip
    marker_tip_x = marker_x - margin_left - map_offset_x
    marker_tip_y = marker_y - margin_top - map_offset_y

    global_px_x = tile_x * tile_size + marker_tip_x
    global_px_y = tile_y * tile_size + marker_tip_y

    # --- Convert to lat/lon
    return pixels_to_latlon(global_px_x, global_px_y, zoom)

def extract_map_center_latlon(driver):
    zoom_to_max(driver)  
    
    tile_img = driver.find_element(By.CSS_SELECTOR, 'img.leaflet-tile')
    tile_url = tile_img.get_attribute("src")
    zoom, tile_x, tile_y = map(int, re.search(r'/(\d+)/(\d+)/(\d+)\.png', tile_url).groups())
    tile_size = 256

    # Find the main map pane transform
    map_pane = driver.find_element(By.CSS_SELECTOR, '.leaflet-map-pane')
    map_transform = map_pane.get_attribute("style")
    map_offset_match = re.search(r'translate3d\(([-\d.]+)px,\s*([-\d.]+)px', map_transform)
    map_offset_x, map_offset_y = map(float, map_offset_match.groups())

    # Get the map container size (in pixels)
    map_container = driver.find_element(By.CSS_SELECTOR, '.leaflet-container')
    width = map_container.size['width']
    height = map_container.size['height']

    # Compute center pixel in global map pixel space
    center_px_x = tile_x * tile_size + width / 2 + map_offset_x
    center_px_y = tile_y * tile_size + height / 2 + map_offset_y

    # Convert to lat/lon
    return center_pixels_to_latlon(center_px_x, center_px_y, zoom)

In [ ]:
def make_hyperlink(value):
    return '=HYPERLINK("%s")' % (value)
def remove_hyperlink(value):
    value = value.replace('=HYPERLINK("', '')
    value = value.replace('")', '')
    return value



In [ ]:
for ind_zona, nome_zona in enumerate(lista_zone):
    nome_zona_pulito = re.sub(r"[^a-zA-Z0-9]+", ' ', nome_zona)
    print ("Zona: {}".format(nome_zona))

    nome_file = 'Visita_Lista_Annunci_Immobiliare_Case_2025_{}.xlsx'.format(nome_zona_pulito)
    print ("Controllo se è presente un file con gli appartamenti della zona già visionati")
    try:
        visto_zona = pd.read_excel(nome_file, sheet_name = "Annunci Completo", index_col=0)
        hyperlinks = get_hyperlinks(nome_file)
        hyperlinks = hyperlinks[:len(visto_zona)]
        visto_zona['Link'] = hyperlinks
        flag_esiste = 1
        print (f"File presente per zona {nome_zona_pulito}")
    except Exception as e:
        print(e)
        print (f"File non presente per zona {nome_zona_pulito}")
        continue
    latitudes = []
    longitudes = []
    visto_zona['Da vedere'] = visto_zona['Da vedere'].fillna("No")

    for i, row in visto_zona.iterrows():
        if row['Da vedere'].lower()=='si':
            if ('Latitude' not in row.index) or (pd.isna(row['Latitude'])) or (row['Latitude']==0):
                print(f"Calcolo Latitude Longitude per {row['Titolo']}")
                driver.get(row['Link'])
                time.sleep(10)
                # Inject JS to get marker lat/lon using Leaflet directly
                close_optional_banner(driver)
                time.sleep(1)
                try:
                    lat, lon = extract_precise_pin_coordinates(driver)
                except:
                    try:
                        lat, lon = extract_map_center_latlon(driver)
                    except:
                        lat, lon = 0, 0
                latitudes.append(lat)
                longitudes.append(lon)
                print(lat, lon)
            else:
                latitudes.append(row['Latitude'] if ('Latitude' in row.index and pd.notna(row['Latitude'])) else 0 )
                longitudes.append(row['Longitudes'] if ('Longitudes' in row.index and pd.notna(row['Longitudes'])) else 0 )
        else:
            latitudes.append(row['Latitude'] if ('Latitude' in row.index and pd.notna(row['Latitude'])) else 0 )
            longitudes.append(row['Longitudes'] if ('Longitudes' in row.index and pd.notna(row['Longitudes'])) else 0 )
    visto_zona['Latitude'] = latitudes
    visto_zona['Longitudes'] = longitudes
    visto_zona['Link'] = visto_zona['Link'].apply(lambda x: make_hyperlink(x))
    
    nome_file = os.path.join(os.getcwd(), 'Visita_Lista_Annunci_Immobiliare_Case_2025_{}.xlsx'.format(nome_zona_pulito))
    writer = pd.ExcelWriter(nome_file, engine='xlsxwriter')
    visto_zona.to_excel(writer, sheet_name='Annunci Completo')
    writer.close()
    print(f"Terminato zona {nome_zona_pulito}")
    print("-"*80)
    
print ("Finito il codice")
driver.quit()

In [ ]:
import plotly.express as px
import pandas as pd

# Your coordinates (from tile conversion or otherwise)

visto_zona_filt = visto_zona[visto_zona['Da vedere'].str.lower()=='si']
visto_zona_filt["Size"] = 30


fig = px.scatter_mapbox(
    visto_zona_filt,
    lat="Latitude",
    lon="Longitudes",
    hover_name="Titolo",
    zoom=16,  # street level
    height=500,
    size="Size",
    color_discrete_sequence=["lightgreen"]
)

# Choose a mapbox style (street maps)
fig.update_layout(
    mapbox_style="open-street-map",  # No Mapbox token required
    margin={"r":0, "t":0, "l":0, "b":0}
)

fig.show()

In [ ]:
visto_zona

In [ ]:
visto_zona_filt